# FedAT: asynchronous tiers (T=3)

Clients are grouped into **tiers of similar speed**. Inside a tier, training is synchronous; the tiers update the global model asynchronously, and slower tiers get larger weights so the model is not pulled toward the fast clients. Based on Chai et al., *FedAT: A High-Performance and Communication-Efficient Federated Learning System with Asynchronous Tiers* (SC 2021).

**How it works**

1. Before training, clients are sorted by their expected response time (download + training + upload) and split into `tiers` tiers, fastest first.
2. Every tier runs its own rounds: its clients start from the latest global model and train; the tier waits for all of them and averages them (FedAvg) into the **tier model**.
3. When a tier finishes a round, the global model becomes a weighted average of all tier models. Tier `m` gets the update count of the mirrored tier `T + 1 - m`, so the slowest tier weighs as much as the fastest tier has updated.
4. Clients also add a constraint term `prox / 2 * ||w - w_global||^2` to their loss, which keeps them close to the global model on non-IID data.

> **Note:** fast tiers update often but are weighted less; slow tiers update rarely but weigh more. This keeps every tier's data in the global model without anyone waiting for the stragglers.

> **Caution:** simplifications and observations compared with the paper:
> * Tiers come from the simulated device speeds and latencies, which is what the paper's profiling measures. The paper also re-profiles clients during training; here the devices do not change, so the tiers are fixed.
> * With the paper's weighting, the first global update gives all the weight to the slowest tier, which has not trained yet, so the global model stays at the start until every tier has reported once. This follows the paper's Algorithm 2 exactly.
> * `prox` is 0.01, not the paper's 0.4. The paper uses Adam as the local optimizer; with the shared SGD settings 0.4 pulls clients back too hard (validation 86.0% vs 91.6%). 0, 0.001 and 0.01 score the same on validation.
> * The model compression (polyline encoding) only reduces communication cost and is not simulated.
> * With 10 clients, every client of a tier trains in every round (`clients_per_round = None`). The paper samples 10 clients per tier round from larger tiers.

## 1. Load the simulator

Gives this notebook everything from `hierarchical_simulator.ipynb`: all of `simulator.ipynb` (settings, `run`, ...) plus `GroupApproach` for group-based approaches.

In [1]:
%run ../hierarchical_simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FedAT(GroupApproach):
    name = "fedat"
    label = "FedAT asynchronous tiers T=3"          # shown in plot legends
    tiers = 3                     # number of tiers (fastest to slowest)
    clients_per_round = None      # clients sampled per tier round; None = the whole tier
    prox = 0.01                   # lambda of the local constraint term (pulls clients toward the global model)

    def form_groups(self, server):
        # expected time of one job (download + training + upload): what FedAT's tiering module profiles
        S = server.S
        cost = S.batch_size / S.speed_batch_size if S.speed_batch_size else 1.0
        def job_time(c):
            batches = math.ceil(c.samples / S.batch_size) * self.local_epochs(server, c)
            return c.download_latency + batches * cost / c.speed + c.upload_latency
        order = [c.id for c in sorted(server.clients, key=job_time)]
        edges = np.linspace(0, len(order), self.tiers + 1).round().astype(int)
        return [order[a:b] for a, b in zip(edges[:-1], edges[1:])]

    def start(self, server):
        self.tier_models = [server.model] * self.tiers
        self.tier_updates = [0] * self.tiers          # how many times each tier updated the global model
        super().start(server)

    def pick_members(self, server, g):
        tier = self.groups[g]
        if self.clients_per_round is None or self.clients_per_round >= len(tier):
            return list(tier)
        return sorted(int(c) for c in server.rng.choice(tier, self.clients_per_round, replace=False))

    def local_loss(self, client, model):
        if not self.prox:
            return None
        return self.prox / 2 * sum(((p - client.start_state[n].to(p.device)) ** 2).sum()
                                   for n, p in model.named_parameters())

    def on_group_done(self, server, g, model, updates):
        self.tier_models[g] = model
        self.tier_updates[g] += 1
        # slower tiers get the update counts of the matching faster tiers, so they weigh more
        weights = [self.tier_updates[self.tiers - 1 - m] for m in range(self.tiers)]
        server.set_model(average_models(self.tier_models, weights), updates)

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedAT(tiers=2))`, or `run(FedAT(prox=0.4))` for the paper's constraint strength. It is saved as a separate run.

In [3]:
results = run(FedAT())

fedat                        seed=42   val= 92.08%  acc= 93.20%  time=   693.2s  updates=  100  worst10%= 92.05%  wall=40.3s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
29,42,693.2295,100,0.932,0.2154,0.9205,3.0
